In [ ]:
# Display side-by-side comparison results
from IPython.display import Image, display
import matplotlib.pyplot as plt
import os

comparison_dir = 'outputs/samples/comparisons'
if os.path.exists(comparison_dir):
    comparison_files = sorted([f for f in os.listdir(comparison_dir) if f.endswith('.png')])
    
    print(f"🖼️  Displaying {len(comparison_files)} inpainting results:")
    print("   Format: Original | Vessel Mask | Inpainted Result")
    print("   " + "="*50)
    
    # Display all comparison images
    for img_file in comparison_files:
        print(f"\n📄 {img_file}")
        display(Image(os.path.join(comparison_dir, img_file)))
        
else:
    print("❌ No comparison visualizations found. Run the cells above first.")

### Display Inpainting Results

In [ ]:
# Run inference using the trained model
print("🔄 Running inference on test samples...")
!make inference

# Run visualization to create side-by-side comparisons
print("\n🎨 Creating visualizations...")
!make visualize

# Check inference results
print("\n📊 Inference results:")
if os.path.exists('outputs/samples/results'):
    result_files = os.listdir('outputs/samples/results')
    print(f"  Generated: {len(result_files)} inpainted images")
    print(f"  Results saved to: outputs/samples/results/")
else:
    print("  ❌ No inference results found.")

In [ ]:
# Prepare test samples from validation set
!make prepare-samples

# Check that samples were created
import os
print("\n📁 Sample files created:")
if os.path.exists('outputs/samples/test_img'):
    img_files = os.listdir('outputs/samples/test_img')
    mask_files = os.listdir('outputs/samples/test_mask') if os.path.exists('outputs/samples/test_mask') else []
    print(f"  Images: {len(img_files)} files in outputs/samples/test_img/")
    print(f"  Masks:  {len(mask_files)} files in outputs/samples/test_mask/")
    print(f"  Sample files: {img_files[:3]}...")
else:
    print("  ❌ No sample files found. Check if ARCADE data is available.")

---
## Inference & Visualization

### Prepare Test Samples and Run Inference

# CMT — ARCADE Inpainting Pipeline

Train the CMT inpainting model on vessel-masked coronary angiography X-rays.

---
⚠️ **Before running:** Runtime → Change runtime type → **T4 GPU**

**Features:**
- Enhanced metrics: PSNR, SSIM, Wasserstein Distance, RMSE
- Improved visualization with adaptive vessel detection
- Automated Google Drive checkpoint mirroring
- Makefile-based workflow

In [1]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi

Wed May  6 10:21:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────
!git clone https://github.com/C0d3Crush/arcade-xray-inpainting.git
%cd /content/arcade-xray-inpainting

Cloning into 'arcade-xray-inpainting'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 240 (delta 25), reused 57 (delta 17), pack-reused 172 (from 1)
Receiving objects: 100% (240/240), 1.11 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/arcade-xray-inpainting


In [3]:
# ── 3. Install dependencies ───────────────────────────────────────────────
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
ERROR: Cannot install -r requirements.txt (line 35) and click==8.1.8 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [11]:
from google.colab import drive
drive.mount('/content/drive')

ARCADE_ZIP = '/content/drive/MyDrive/CMT/arcade.zip'

!unzip -q -o "$ARCADE_ZIP" -d /content/arcade-xray-inpainting/
!find arcade -name '*.json' | head -5

# Check extracted structure
!ls -la arcade/syntax/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
arcade/syntax/train/annotations/train.json
arcade/syntax/val/annotations/val.json
arcade/syntax/test/annotations/test.json
total 20
drwxrwxr-x 5 root root 4096 Dec 15  2023 .
drwxr-xr-x 3 root root 4096 May  6 10:26 ..
drwxrwxr-x 4 root root 4096 Dec 15  2023 test
drwxrwxr-x 4 root root 4096 Dec 15  2023 train
drwxrwxr-x 4 root root 4096 Dec 15  2023 val


---
## Setup & Preprocessing

### Cache Masks (Optional - Speeds up training)

In [12]:
import os

# Create the masks_cache directory if it doesn't exist
os.makedirs('masks_cache/train', exist_ok=True)
os.makedirs('masks_cache/val', exist_ok=True)

# ── 5. Cache masks and annotations for faster training ────────────────────
print("Caching train masks...")
!python scripts/cache_masks.py --annotations arcade/syntax/train/annotations/train.json --images arcade/syntax/train/images --output masks_cache/train

print("Caching val masks...")
!python scripts/cache_masks.py --annotations arcade/syntax/val/annotations/val.json --images arcade/syntax/val/images --output masks_cache/val

Caching train masks...
Loading annotations from arcade/syntax/train/annotations/train.json...
  Found 1000 images
  1000 images with vessel annotations

Generating masks → masks_cache/train/
Processing: 100% 1000/1000 [00:00<00:00, 251110.82it/s]

✓ Cached 1000 masks to masks_cache/train/

Usage:
  python train.py --train_mask masks_cache/train ...
Caching val masks...
Loading annotations from arcade/syntax/val/annotations/val.json...
  Found 200 images
  200 images with vessel annotations

Generating masks → masks_cache/val/
Processing: 100% 200/200 [00:00<00:00, 234777.72it/s]

✓ Cached 200 masks to masks_cache/val/

Usage:
  python train.py --train_mask masks_cache/val ...


---
## Train CMT Inpainting Model

### Setup Google Drive Mirroring (Optional)

### Patch Training Mode

**NEW:** Patch-based training extracts random patches from full-resolution images instead of downscaling:

- `--patch_mode` - Extract 64×64 patches from original 512×512+ images
- `--patches_per_image 16` - 16 random patches per image → 16× more training data  
- Preserves fine details and vessel boundaries
- Better learning from high-resolution data

**Comparison:**
- **Patch Training:** 1000 images → 16,000 patches (recommended)
- **Original:** 1000 images → 1000 resized samples

In [ ]:
# ── 6a. Setup Google Drive checkpoint mirroring ──────────────────────────
!mkdir -p /content/drive/MyDrive/CMT/checkpoints

# ── 6b. Train CMT with PATCH MODE (extracts patches from full resolution) ────
!python src/train.py \
    --train_img  arcade/syntax/train/images \
    --train_ann  arcade/syntax/train/annotations/train.json \
    --val_img    arcade/syntax/val/images \
    --val_ann    arcade/syntax/val/annotations/val.json \
    --patch_mode \
    --input_size 64 \
    --patches_per_image 16 \
    --epochs     100 \
    --batch_size 16 \
    --device     cuda

# Alternative: Original resize training (downscales entire image)
# !python src/train.py --input_size 64 --epochs 100 --batch_size 16 --device cuda

# Alternative: Use Makefile
# !make train ARGS="--patch_mode --input_size 64 --patches_per_image 16 --epochs 50 --device cuda"

---
## Results & Visualization

### Plot Training Metrics

In [ ]:
# ── 7a. Plot training metrics ──────────────────────────────────────────────
!python scripts/plot_training.py checkpoints/training_log.csv

# Display the plot
from IPython.display import Image, display
display(Image('training_plot.png'))

# ── 7b. Generate test samples and visualizations ───────────────────────────
!make prepare-samples
!make inference
!make visualize

# Display comparison results
import os
comparison_dir = 'outputs/samples/comparisons'
if os.path.exists(comparison_dir):
    for img_file in sorted(os.listdir(comparison_dir))[:3]:  # Show first 3
        if img_file.endswith('.png'):
            print(f"\n--- {img_file} ---")
            display(Image(os.path.join(comparison_dir, img_file)))

---
## Download Checkpoint

### Download from Colab

In [ ]:
# ── 8. Download best checkpoint ───────────────────────────────────────────
from google.colab import files

# Download checkpoint (automatically mirrored to Google Drive)
files.download('checkpoints/best.pth')
files.download('checkpoints/training_log.csv')

print("\n✓ Checkpoint also available in Google Drive: /MyDrive/CMT/checkpoints/")

### Training Metrics

The model tracks comprehensive evaluation metrics:
- **PSNR** (Peak Signal-to-Noise Ratio) - Image quality
- **SSIM** (Structural Similarity Index) - Perceptual quality  
- **Wasserstein Distance** - Distribution similarity (Earth Mover's Distance)
- **RMSE** (Root Mean Square Error) - Pixel-level accuracy